# 📖 Notebook 3: Search Ranking & Relevance

Finding businesses near you is only half the problem. The other half is **ranking** them so the best results appear first. When you search "pizza" on Yelp, hundreds of places might match — but you want the delicious, nearby, highly-rated ones at the top.

This notebook explores how Elasticsearch scores and ranks search results by combining **text relevance**, **distance**, and **business quality** signals.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Elasticsearch scores text relevance using TF-IDF / BM25
- How to boost results by rating, review count, and distance
- How to build a multi-signal ranking function
- How full-text search with fuzzy matching handles typos
- How to use Elasticsearch `function_score` for custom ranking

## 🛠️ Setup

```bash
cd 06-system-designs/yelp
docker compose up -d
```

**Important**: Run Notebook 1 first to index businesses into Elasticsearch, or run the setup cell below to re-index.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "yelp_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}
ES_URL = "http://localhost:9200"
INDEX_NAME = "businesses"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def get_es():
    return Elasticsearch(ES_URL)

es = get_es()
r = get_redis()

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis: {e}")

try:
    info = es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}")

In [ ]:
# Ensure the ES index exists (re-index if needed)

if not es.indices.exists(index=INDEX_NAME):
    print("Index not found — creating and loading data...")

    es.indices.create(index=INDEX_NAME, body={
        "mappings": {
            "properties": {
                "name":        {"type": "text", "analyzer": "standard"},
                "description": {"type": "text"},
                "city":        {"type": "keyword"},
                "category":    {"type": "keyword"},
                "location":    {"type": "geo_point"},
                "avg_rating":  {"type": "float"},
                "num_reviews": {"type": "integer"},
                "price_range": {"type": "integer"}
            }
        }
    })

    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT b.id, b.name, b.description, b.city, b.latitude, b.longitude,
               b.avg_rating, b.num_reviews, b.price_range, c.name AS category
        FROM businesses b JOIN categories c ON b.category_id = c.id
    """)
    businesses = cur.fetchall()
    conn.close()

    actions = [{
        "_index": INDEX_NAME, "_id": biz["id"],
        "_source": {
            "name": biz["name"], "description": biz["description"],
            "city": biz["city"], "category": biz["category"],
            "location": {"lat": float(biz["latitude"]), "lon": float(biz["longitude"])},
            "avg_rating": float(biz["avg_rating"] or 0),
            "num_reviews": biz["num_reviews"] or 0,
            "price_range": biz["price_range"]
        }
    } for biz in businesses]

    success, _ = bulk(es, actions)
    es.indices.refresh(index=INDEX_NAME)
    print(f"✅ Indexed {success} businesses")
else:
    count = es.count(index=INDEX_NAME)["count"]
    print(f"✅ ES index exists with {count} businesses")

## 🔀 The Index Is a Copy, and Copies Go Stale

Everything below this point ranks on `avg_rating` and `num_reviews` **as stored in
Elasticsearch**. Those are copies. The source of truth is Postgres, and nothing in this lab
pushes a change from one to the other — Notebook 2 wrote dozens of reviews straight to Postgres
and Elasticsearch never heard about any of them.

This is not a flaw in the toy; it is the actual reason "keep Elasticsearch in sync with Postgres"
is a whole box on the architecture diagram. The failure is invisible from the search side: the
query is fast, the results look plausible, and the star rating on the card is simply the wrong
number.

Three ways real systems close the gap, cheapest first:

| Approach | Lag | Cost | Fails when |
|----------|-----|------|-----------|
| **Dual write** (app writes both) | ~0 | trivial | the second write fails and nothing retries — silent, permanent divergence |
| **Change Data Capture** (tail the WAL, e.g. Debezium) | seconds | an extra service | the connector falls behind or the WAL slot fills |
| **Periodic full reconcile** (batch re-index) | hours | a nightly job | never — which is why you run it *even when* you have CDC |

The audit below is that third one, in miniature: read both sides, diff them, re-index whatever
disagrees. Notice we deliberately create a fresh divergence first, so this cell reproduces the
problem on every run rather than only when you happen to have run Notebook 2.

In [ ]:
def index_audit():
    """Diff the Elasticsearch copy against Postgres. Returns the ids that disagree."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("SELECT id, avg_rating, num_reviews FROM businesses")
    truth = {row["id"]: (float(row["avg_rating"]), row["num_reviews"]) for row in cur.fetchall()}
    conn.close()

    es.indices.refresh(index=INDEX_NAME)
    resp = es.search(index=INDEX_NAME, body={
        "query": {"match_all": {}}, "size": 10_000,
        "_source": ["avg_rating", "num_reviews"],
    })
    indexed = {int(h["_id"]): (h["_source"]["avg_rating"], h["_source"]["num_reviews"])
               for h in resp["hits"]["hits"]}

    stale = []
    for bid, (pg_avg, pg_n) in truth.items():
        if bid not in indexed:
            stale.append(bid)
            continue
        es_avg, es_n = indexed[bid]
        # avg_rating is a float32 in Lucene, so compare with a tolerance well under one step
        # of DECIMAL(3,2). num_reviews is an integer and must match exactly.
        if abs(pg_avg - es_avg) > 0.005 or pg_n != es_n:
            stale.append(bid)
    return sorted(stale), truth


def reindex(business_ids):
    """The catch-up write. In production this is what the CDC consumer does per event."""
    if not business_ids:
        return 0
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT b.id, b.name, b.description, b.city, b.latitude, b.longitude,
               b.avg_rating, b.num_reviews, b.price_range, c.name AS category
        FROM businesses b JOIN categories c ON b.category_id = c.id
        WHERE b.id = ANY(%s)
    """, (list(business_ids),))
    rows = cur.fetchall()
    conn.close()

    actions = [{
        "_index": INDEX_NAME, "_id": biz["id"],
        "_source": {
            "name": biz["name"], "description": biz["description"],
            "city": biz["city"], "category": biz["category"],
            "location": {"lat": float(biz["latitude"]), "lon": float(biz["longitude"])},
            "avg_rating": float(biz["avg_rating"] or 0),
            "num_reviews": biz["num_reviews"] or 0,
            "price_range": biz["price_range"],
        }
    } for biz in rows]
    ok, _ = bulk(es, actions)
    es.indices.refresh(index=INDEX_NAME)
    return ok


# --- 1. What has already drifted? (non-zero if you ran Notebook 2 against this index) ---
stale, truth = index_audit()
print(f"🔎 Audit before we touch anything: {len(stale)} of {len(truth)} businesses disagree")
if stale:
    print(f"   e.g. business ids {stale[:8]}{' ...' if len(stale) > 8 else ''}")
    print("   Those are Notebook 2's reviews. Postgres has them; the search index does not.")

# --- 2. Create a fresh divergence, so this lesson reproduces every run ---
DIVERGE_BIZ = 200

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Notebook 2 adds `rating_sum` (and `db/init.sql` creates it on a fresh database). Make sure it
# is there so this notebook runs standalone, and so our write leaves the row self-consistent.
cur.execute("ALTER TABLE businesses ADD COLUMN IF NOT EXISTS rating_sum INTEGER NOT NULL DEFAULT 0;")

cur.execute("""
    SELECT id FROM users
    WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = %s)
    ORDER BY id LIMIT 1;
""", (DIVERGE_BIZ,))
reviewer = cur.fetchone()["id"]
cur.execute("""
    INSERT INTO reviews (business_id, user_id, rating, text)
    VALUES (%s, %s, 1, 'Search-index divergence demo');
""", (DIVERGE_BIZ, reviewer))

# Recompute this business's summary from `reviews` in the same transaction. The point of this
# section is what Elasticsearch does NOT see, so we keep the Postgres side above suspicion.
cur.execute("""
    UPDATE businesses b
    SET rating_sum  = s.total,
        num_reviews = s.cnt,
        avg_rating  = ROUND(s.total::numeric / s.cnt, 2),
        updated_at  = NOW()
    FROM (SELECT SUM(rating) AS total, COUNT(*) AS cnt
          FROM reviews WHERE business_id = %s) s
    WHERE b.id = %s
    RETURNING b.avg_rating, b.num_reviews;
""", (DIVERGE_BIZ, DIVERGE_BIZ))
pg_after = cur.fetchone()
conn.commit()
conn.close()

es.indices.refresh(index=INDEX_NAME)
es_doc = es.get(index=INDEX_NAME, id=DIVERGE_BIZ)["_source"]

print(f"\n🕳️  Wrote a 1★ review for business {DIVERGE_BIZ} through Postgres only:")
print(f"   Postgres      : ⭐{float(pg_after['avg_rating']):.2f} ({pg_after['num_reviews']} reviews)")
print(f"   Elasticsearch : ⭐{es_doc['avg_rating']:.2f} ({es_doc['num_reviews']} reviews)  ← what users see")

assert es_doc["num_reviews"] != pg_after["num_reviews"], (
    "expected Elasticsearch to be behind Postgres after a Postgres-only write — if it is not, "
    "something is already syncing and this demonstration no longer teaches anything")

stale, truth = index_audit()
assert DIVERGE_BIZ in stale, f"the audit failed to spot the divergence it just created"
print(f"\n🔎 Audit now reports {len(stale)} stale documents, including {DIVERGE_BIZ}.")

# --- 3. Reconcile, and prove it converged ---
reindexed = reindex(stale)
print(f"🔄 Re-indexed {reindexed} documents.")

stale_after, _ = index_audit()
print(f"🔎 Audit after reconcile: {len(stale_after)} stale documents")
assert stale_after == [], (
    f"reconciliation left {len(stale_after)} documents disagreeing with Postgres: {stale_after[:10]}")

print("\n✅ Elasticsearch and Postgres now agree — every ranking below is scoring on real data.")
print("   In production this loop runs continuously (CDC) with the batch audit as a safety net.")

## 🔤 Full-Text Search: How Elasticsearch Scores Text

When you search for "restaurant," Elasticsearch uses **BM25** (an improved version of TF-IDF) to score how relevant each document is:

- **TF (Term Frequency)**: How often does "restaurant" appear in this business name? More = more relevant.
- **IDF (Inverse Document Frequency)**: How rare is "restaurant" across all businesses? Rarer terms get higher scores.
- **Field length**: Shorter fields get a boost — a business named "Restaurant" scores higher than "The Best Downtown Restaurant And Bar Serving Food".

Let's see BM25 scoring in action.

In [ ]:
# Basic text search — Elasticsearch assigns a relevance score to each result

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "match": {
                "name": "restaurant"
            }
        },
        "size": 10
    }
)

print(f"🔍 Search: 'restaurant'")
print(f"   Total matches: {result['hits']['total']['value']}\n")
print(f"   {'Score':>8}  {'Rating':>6}  {'Reviews':>7}  Name")
print(f"   {'-'*60}")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    print(f"   {hit['_score']:>8.3f}  ⭐{src['avg_rating']:>4.1f}  {src['num_reviews']:>7}  {src['name']}")

print(f"\n💡 Notice: results are sorted by text relevance (BM25 score), not by rating!")
print(f"   A highly-rated business might be buried if its name doesn't match well.")

## 🔍 Fuzzy Matching: Handling Typos

Users make typos: "resturant", "caffee", "fitnes". Elasticsearch's fuzzy matching uses **edit distance** (Levenshtein distance) to find matches within 1-2 character changes.

In [ ]:
# Search with a typo — without fuzzy matching
result_strict = es.search(
    index=INDEX_NAME,
    body={"query": {"match": {"name": "resturant"}}, "size": 5}
)

# Search with fuzzy matching enabled
result_fuzzy = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "match": {
                "name": {
                    "query": "resturant",
                    "fuzziness": "AUTO"  # allows 1-2 character edits based on word length
                }
            }
        },
        "size": 5
    }
)

print(f"🔍 Search for 'resturant' (a common typo)\n")
print(f"   Without fuzzy: {result_strict['hits']['total']['value']} results")
print(f"   With fuzzy:    {result_fuzzy['hits']['total']['value']} results\n")

if result_fuzzy["hits"]["hits"]:
    print(f"   Top fuzzy matches:")
    for hit in result_fuzzy["hits"]["hits"][:5]:
        print(f"   📍 score={hit['_score']:.3f} | {hit['_source']['name']}")

print(f"\n💡 Fuzzy matching catches typos so users still find what they want.")

## 🏆 Custom Ranking with function_score

Real search ranking isn't just text relevance. Yelp combines multiple signals:

| Signal | Why It Matters |
|--------|---------------|
| **Text match** | The name/description should match the query |
| **Distance** | Closer businesses are more useful |
| **Average rating** | Higher-rated businesses should rank higher |
| **Review count** | More reviews = more trustworthy rating |

Elasticsearch's `function_score` query lets us combine all of these into a single ranking formula.

In [ ]:
# Multi-signal ranking: combine text relevance + rating + distance

user_lat, user_lon = 40.758, -73.985  # Manhattan

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "function_score": {
                "query": {
                    "bool": {
                        "must": [
                            {"match": {"name": {"query": "restaurant", "fuzziness": "AUTO"}}}
                        ],
                        "filter": [
                            {"geo_distance": {"distance": "10km", "location": {"lat": user_lat, "lon": user_lon}}}
                        ]
                    }
                },
                "functions": [
                    {
                        # Boost by average rating. `log1p` is log10(1 + factor * value), so a
                        # 5-star business contributes log10(1 + 0.2*5) = 0.301, and weight 3
                        # turns that into 0.90. A 0-star business contributes exactly 0.
                        "field_value_factor": {
                            "field": "avg_rating",
                            "factor": 0.2,       # how strongly rating affects score
                            "modifier": "log1p",  # smooth the curve
                            "missing": 1          # default if the field is absent
                        },
                        "weight": 3
                    },
                    {
                        # Boost by number of reviews (more reviews = more trustworthy).
                        # log10(1 + 0.1*n): 6 reviews → 0.20, 100 reviews → 1.04. The log is what
                        # stops a business with 10,000 reviews from burying everything else.
                        "field_value_factor": {
                            "field": "num_reviews",
                            "factor": 0.1,
                            "modifier": "log1p",
                            "missing": 1
                        },
                        "weight": 1
                    },
                    {
                        # Decay score based on distance (closer = higher score).
                        # Read these three together: no decay inside `offset`, and the curve
                        # reaches `decay` at offset + scale — so 50% at 2.5 km, NOT at 2 km.
                        # That off-by-one-parameter is the classic function_score mistake.
                        "gauss": {
                            "location": {
                                "origin": {"lat": user_lat, "lon": user_lon},
                                "scale": "2km",    # 50% is reached `scale` PAST the offset
                                "offset": "500m",  # no decay within 500m
                                "decay": 0.5
                            }
                        },
                        "weight": 2
                    }
                ],
                "score_mode": "sum",      # combine function scores by adding them
                "boost_mode": "multiply"  # multiply function scores with query score
            }
        },
        "size": 10
    }
)

print(f"🏆 Multi-Signal Ranking: 'restaurant' near Manhattan")
print(f"   score = bm25 × ( 3·log10(1+0.2·rating) + 1·log10(1+0.1·reviews) + 2·gauss(distance) )\n")
print(f"   {'Score':>8}  {'Rating':>6}  {'Reviews':>7}  {'City':<15}  Name")
print(f"   {'-'*70}")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    print(f"   {hit['_score']:>8.2f}  ⭐{src['avg_rating']:>4.1f}  {src['num_reviews']:>7}  {src['city']:<15}  {src['name']}")

print(f"\n💡 Now the ranking considers rating, review count, AND distance — not just text match!")

### 🔬 Check the Arithmetic

`function_score` is easy to write and hard to reason about, and the failure mode is silent: a
mis-set `offset` or a `factor` an order of magnitude off doesn't error, it just ranks slightly
wrong forever. So let's recompute the score ourselves and check it against what Elasticsearch
returned.

The pieces, exactly as Elasticsearch defines them:

- `field_value_factor` with `modifier: log1p` is **log base 10** of `1 + factor × value` — not the
  natural log. Getting this wrong changes every boost by a factor of 2.30.
- `gauss(d) = exp( -max(0, d - offset)² / (2σ²) )` with `σ² = -scale² / (2·ln(decay))`.
  Substituting `d = offset + scale` gives exactly `decay`, which is the check that the 50% point
  sits at 2.5 km and not 2 km.
- `weight` alongside a function **multiplies** that function's output.
- `score_mode: "sum"` adds the weighted functions; `boost_mode: "multiply"` multiplies the total
  by the BM25 score of the inner query.

To get the BM25 term on its own we run the identical `bool` query *without* the `function_score`
wrapper and read `_score` from that.

In [ ]:
import math

def gauss_decay(distance_m, scale_m, offset_m, decay):
    """Elasticsearch's gaussian decay function, written out."""
    sigma_sq = -(scale_m ** 2) / (2 * math.log(decay))
    excess = max(0.0, distance_m - offset_m)
    return math.exp(-(excess ** 2) / (2 * sigma_sq))

def arc_distance_m(lat1, lon1, lat2, lon2):
    """Haversine on Lucene's mean-Earth sphere — what ES uses for geo decay by default."""
    R = 6_371_008.7714
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin((p2 - p1) / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))

# Sanity check the decay curve against its own definition before trusting it on real documents.
assert abs(gauss_decay(2500, 2000, 500, 0.5) - 0.5) < 1e-9, "gauss must hit `decay` at offset+scale"
assert gauss_decay(500, 2000, 500, 0.5) == 1.0, "no decay inside the offset"
assert gauss_decay(2000, 2000, 500, 0.5) > 0.5, (
    "at 2 km the score should still be ABOVE 50% — the 50% point is offset+scale = 2.5 km")
print(f"📐 gauss at 500 m: {gauss_decay(500, 2000, 500, 0.5):.4f}   "
      f"2 km: {gauss_decay(2000, 2000, 500, 0.5):.4f}   "
      f"2.5 km: {gauss_decay(2500, 2000, 500, 0.5):.4f}   "
      f"5 km: {gauss_decay(5000, 2000, 500, 0.5):.4f}")

INNER_QUERY = {
    "bool": {
        "must": [{"match": {"name": {"query": "restaurant", "fuzziness": "AUTO"}}}],
        "filter": [{"geo_distance": {"distance": "10km", "location": {"lat": user_lat, "lon": user_lon}}}]
    }
}

# BM25 alone, no function_score wrapper. Ask for every match, not just the top 20 — the
# boosted ranking reorders things, so a doc in the boosted top 20 need not be in the plain one.
plain = es.search(index=INDEX_NAME, body={"query": INNER_QUERY, "size": 1000, "_source": False})
bm25 = {h["_id"]: h["_score"] for h in plain["hits"]["hits"]}
print(f"🔤 {len(bm25)} documents match the inner query; reconciling the top 20 after boosting.")

# The same query with the ranking functions attached (identical to the cell above).
scored = es.search(index=INDEX_NAME, body={
    "query": {"function_score": {
        "query": INNER_QUERY,
        "functions": [
            {"field_value_factor": {"field": "avg_rating", "factor": 0.2, "modifier": "log1p", "missing": 1}, "weight": 3},
            {"field_value_factor": {"field": "num_reviews", "factor": 0.1, "modifier": "log1p", "missing": 1}, "weight": 1},
            {"gauss": {"location": {"origin": {"lat": user_lat, "lon": user_lon},
                                    "scale": "2km", "offset": "500m", "decay": 0.5}}, "weight": 2},
        ],
        "score_mode": "sum", "boost_mode": "multiply",
    }},
    "size": 20,
    "_source": ["name", "avg_rating", "num_reviews", "location"],
})

print(f"\n{'rating':>7} {'reviews':>8} {'dist_km':>8} {'bm25':>7} {'ours':>8} {'ES':>8}  name")
print("   " + "-" * 78)

worst_rel_err = 0.0
checked = 0
for hit in scored["hits"]["hits"]:
    src = hit["_source"]
    assert hit["_id"] in bm25, "every boosted hit must also match the un-boosted inner query"
    loc = src["location"]
    d_m = arc_distance_m(user_lat, user_lon, loc["lat"], loc["lon"])

    rating_term = 3 * math.log10(1 + 0.2 * src["avg_rating"])
    review_term = 1 * math.log10(1 + 0.1 * src["num_reviews"])
    decay_term = 2 * gauss_decay(d_m, 2000, 500, 0.5)
    ours = bm25[hit["_id"]] * (rating_term + review_term + decay_term)

    worst_rel_err = max(worst_rel_err, abs(ours - hit["_score"]) / hit["_score"])
    checked += 1
    print(f"   {src['avg_rating']:>7.2f} {src['num_reviews']:>8} {d_m/1000:>8.2f} "
          f"{bm25[hit['_id']]:>7.3f} {ours:>8.3f} {hit['_score']:>8.3f}  {src['name']}")

print(f"\n   documents checked: {checked}, worst relative error: {worst_rel_err * 100:.4f}%")

assert checked >= 5, f"expected to reconcile at least 5 documents, only got {checked}"
# Lucene scores in float32 and quantises geo_points to ~1 cm, so allow a little slack — but not
# enough to hide a wrong log base (2.30x off) or a misread offset.
assert worst_rel_err < 0.02, (
    f"our formula disagrees with Elasticsearch by {worst_rel_err * 100:.2f}% — "
    "the ranking function is not what the prose says it is")

print("\n💡 If you can rebuild the score by hand, you can debug a ranking complaint.")
print("   If you can't, `function_score` is a black box you are shipping to production.")

## 📊 Comparing Ranking Strategies

Let's compare three ranking strategies side by side to see how they produce different results.

In [ ]:
def search_text_only(query, city=None, size=5):
    """Rank by text relevance only."""
    body = {"query": {"match": {"name": query}}, "size": size}
    if city:
        body["query"] = {"bool": {"must": [{"match": {"name": query}}], "filter": [{"term": {"city": city}}]}}
    return es.search(index=INDEX_NAME, body=body)

def search_rating_sorted(query, city=None, size=5):
    """Match by text, sort by rating."""
    body = {"query": {"match": {"name": query}}, "sort": [{"avg_rating": "desc"}], "size": size}
    if city:
        body["query"] = {"bool": {"must": [{"match": {"name": query}}], "filter": [{"term": {"city": city}}]}}
    return es.search(index=INDEX_NAME, body=body)

def search_multi_signal(query, lat, lon, size=5):
    """Combine text relevance + rating + distance."""
    return es.search(index=INDEX_NAME, body={
        "query": {
            "function_score": {
                "query": {"bool": {
                    "must": [{"match": {"name": {"query": query, "fuzziness": "AUTO"}}}],
                    "filter": [{"geo_distance": {"distance": "10km", "location": {"lat": lat, "lon": lon}}}]
                }},
                "functions": [
                    {"field_value_factor": {"field": "avg_rating", "factor": 0.2, "modifier": "log1p", "missing": 1}, "weight": 3},
                    {"field_value_factor": {"field": "num_reviews", "factor": 0.1, "modifier": "log1p", "missing": 1}, "weight": 1},
                    {"gauss": {"location": {"origin": {"lat": lat, "lon": lon}, "scale": "2km", "offset": "500m", "decay": 0.5}}, "weight": 2}
                ],
                "score_mode": "sum", "boost_mode": "multiply"
            }
        },
        "size": size
    })


def show_results(label, result):
    print(f"\n{label}")
    for i, hit in enumerate(result["hits"]["hits"], 1):
        src = hit["_source"]
        score = hit['_score'] if hit['_score'] else 0
        print(f"   {i}. score={score:>6.2f} | ⭐{src['avg_rating']:>4.1f} ({src['num_reviews']:>2} reviews) | {src['city']:<15} | {src['name']}")


query = "cafe"
print(f"🔍 Searching for '{query}' — three different ranking strategies:\n")

show_results("📝 Strategy 1: Text Relevance Only (BM25)", search_text_only(query))
show_results("⭐ Strategy 2: Sort by Rating", search_rating_sorted(query))
show_results("🏆 Strategy 3: Multi-Signal (text + rating + distance)", search_multi_signal(query, 40.758, -73.985))

print(f"\n💡 Multi-signal ranking balances relevance, quality, and proximity.")
print(f"   This is what real search engines like Yelp actually do!")
print(f"\n⚠️  Read this comparison carefully: strategies 1 and 2 search all five cities, while")
print(f"   strategy 3 also filters to 10 km of Manhattan. It is not a like-for-like ranking")
print(f"   contest — the geo filter is doing some of the work you might credit to the scoring.")

## 🏙️ Search by City Name (Named Locations)

Users often search by city name ("pizza in San Francisco") rather than lat/lon. Yelp maps location names to polygons.

In our simplified version, we use the `city` field as a keyword filter.

In [ ]:
# Search: "fitness" in Chicago

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "must": [{"match": {"name": {"query": "fitness", "fuzziness": "AUTO"}}}],
                "filter": [{"term": {"city": "Chicago"}}]
            }
        },
        "sort": [{"avg_rating": "desc"}],
        "size": 10
    }
)

print(f"🏙️ 'fitness' in Chicago")
print(f"   Found {result['hits']['total']['value']} results\n")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    price = '$' * (src.get('price_range') or 1)
    print(f"   ⭐ {src['avg_rating']:.1f} ({src['num_reviews']} reviews) | {price} | {src['name']}")

print(f"\n💡 In production, Yelp maps 'Chicago' to a polygon and uses geo_shape queries.")
print(f"   For neighborhoods ('The Mission', 'SoHo'), they pre-compute which businesses")
print(f"   are in each area and store location tags on each business document.")

## ⌨️ Autocomplete (Search-As-You-Type)

When you start typing `"piz"` into the Yelp search box, suggestions appear **before you finish the word**. This is called **autocomplete** (or *search-as-you-type*) and it's one of the first things users interact with.

The trick: instead of full-word matching, we match on **prefixes**. Elasticsearch has a few ways to do this; the simplest is the `match_phrase_prefix` query, which treats the **last token** as a prefix.

**Why not just `LIKE 'piz%'` in SQL?**

- Autocomplete needs to be *fast* (< 50 ms) because it fires on every keystroke.
- `LIKE 'piz%'` works with a B-tree, but only on the leading prefix of the full field — not on *words inside* the name.
- `"The Best Pizza"` would not match `LIKE 'piz%'` (starts with "The"), but users expect it to.
- Elasticsearch indexes each **word** independently, so "Pizza" is findable no matter where it sits in the name.

For production at Yelp's scale you'd use dedicated features like `search_as_you_type` fields or the *completion suggester* (which builds a special in-memory data structure). We'll stick with `match_phrase_prefix` here — it's zero-config and good enough to see the idea.


In [ ]:
# Autocomplete: suggest businesses as the user types

def autocomplete(prefix, limit=5):
    """Return top business names that match a partial query like 'piz' or 'coffe'."""
    result = es.search(index=INDEX_NAME, body={
        "query": {
            "match_phrase_prefix": {
                "name": {"query": prefix, "max_expansions": 20}
            }
        },
        # Tiebreak by how well-known the business is (more reviews = more likely what the user meant)
        "sort": ["_score", {"num_reviews": "desc"}],
        "size": limit,
        # We only need the name for the dropdown — tell ES to skip everything else
        "_source": ["name", "city", "avg_rating", "num_reviews"],
    })
    return [hit["_source"] for hit in result["hits"]["hits"]]


# Simulate a user typing "res" → "rest" → "resta" one letter at a time
typed = ""
for ch in "resta":
    typed += ch
    start = time.time()
    suggestions = autocomplete(typed)
    elapsed_ms = (time.time() - start) * 1000
    print(f"⌨️  User typed '{typed}' → {elapsed_ms:>5.1f} ms → {len(suggestions)} suggestions")
    for s in suggestions[:3]:
        print(f"       • {s['name']} ({s['city']}) — ⭐{s['avg_rating']:.1f}, {s['num_reviews']} reviews")
    print()

print("💡 Each keystroke fires a fresh query, but each one is still fast because")
print("   Elasticsearch indexed every word in every business name.")


## ⚡ Caching Popular Search Results

Popular searches ("restaurants in New York", "coffee in San Francisco") are repeated thousands of times per minute. We cache them in Redis.

In [ ]:
def ranked_search_cached(query, lat, lon, radius_km=10, category=None, limit=10):
    """
    Full-featured search with multi-signal ranking and Redis caching.
    This is what a real Yelp search API endpoint might look like.
    """
    # Build cache key from search parameters
    cache_key = f"ranked:{query}:{round(lat,2)}:{round(lon,2)}:{radius_km}:{category or 'all'}:{limit}"

    # Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True

    # Build ES query
    must_clauses = [{"match": {"name": {"query": query, "fuzziness": "AUTO"}}}]
    if category:
        must_clauses.append({"term": {"category": category}})

    result = es.search(index=INDEX_NAME, body={
        "query": {
            "function_score": {
                "query": {"bool": {
                    "must": must_clauses,
                    "filter": [{"geo_distance": {"distance": f"{radius_km}km", "location": {"lat": lat, "lon": lon}}}]
                }},
                "functions": [
                    {"field_value_factor": {"field": "avg_rating", "factor": 0.2, "modifier": "log1p", "missing": 1}, "weight": 3},
                    {"field_value_factor": {"field": "num_reviews", "factor": 0.1, "modifier": "log1p", "missing": 1}, "weight": 1},
                    {"gauss": {"location": {"origin": {"lat": lat, "lon": lon}, "scale": "2km", "offset": "500m", "decay": 0.5}}, "weight": 2}
                ],
                "score_mode": "sum", "boost_mode": "multiply"
            }
        },
        "size": limit
    })

    # Format results
    results = []
    for hit in result["hits"]["hits"]:
        src = hit["_source"]
        results.append({
            "name": src["name"], "city": src["city"], "category": src["category"],
            "avg_rating": src["avg_rating"], "num_reviews": src["num_reviews"],
            "price_range": src.get("price_range", 1),
            "score": round(hit["_score"], 2)
        })

    # Cache for 30 seconds
    r.setex(cache_key, 30, json.dumps(results))
    return results, False


# Benchmark: cold vs cached search.
# Delete only our own keys — this Redis is shared with the other notebooks in this lab
# (and, if you're running several labs, with them too). `flushdb()` here would wipe them.
stale_keys = r.keys("ranked:*")
if stale_keys:
    r.delete(*stale_keys)

start = time.time()
results, from_cache = ranked_search_cached("restaurant", 40.758, -73.985, radius_km=5)
t1 = (time.time() - start) * 1000
print(f"🔍 Cold search: {t1:.2f} ms (cached: {from_cache})")

start = time.time()
results, from_cache = ranked_search_cached("restaurant", 40.758, -73.985, radius_km=5)
t2 = (time.time() - start) * 1000
print(f"⚡ Cached search: {t2:.2f} ms (cached: {from_cache})")
print(f"\n🚀 Cache speedup: {t1/t2:.1f}×\n")

# A Redis GET does no query work at all, so it has to beat a round trip into
# Elasticsearch. If it does not, the cache is not being hit and the demo is broken.
assert from_cache is True, "the second call should have been served from Redis"
assert t2 < t1, f"expected the cached read to be faster, got cold={t1:.2f} ms cached={t2:.2f} ms"

print(f"Top results:")
for i, biz in enumerate(results[:5], 1):
    price = '$' * biz['price_range']
    print(f"   {i}. score={biz['score']:>6.2f} | ⭐{biz['avg_rating']:>4.1f} ({biz['num_reviews']} reviews) | {price:<4} | {biz['name']}")

## 🧹 Cleanup

In [ ]:
# Clean up Redis cache and Elasticsearch index
r = get_redis()
keys = r.keys("ranked:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned {len(keys)} Redis cache keys")

es = get_es()
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"🧹 Deleted ES index '{INDEX_NAME}'")

print("🧹 Cleanup complete!")

## 📚 Summary

### Key Takeaways

1. **BM25 text scoring** gives a baseline relevance score — but it's not enough alone
2. **Fuzzy matching** catches typos using edit distance (Levenshtein)
3. **Multi-signal ranking** combines text relevance, rating, review count, and distance
4. **`function_score`** in Elasticsearch lets you build custom ranking formulas
5. **Redis caching** makes popular searches near-instant (sub-millisecond)
6. **Named locations** (cities, neighborhoods) are mapped to polygons for area-based search
7. **Know your scoring formula by hand** — `log1p` is log base 10, and a gaussian decay reaches
   `decay` at `offset + scale`, not at `scale`. A wrong parameter never errors, it just ranks
   slightly wrong forever
8. **The search index is a copy** — it drifts the moment anything writes to Postgres without
   telling it, and the drift is invisible from the search side. Run a reconcile audit even if
   you have CDC

### System Design Interview Tips

- **Elasticsearch vs Postgres**: Use ES for full-text search at scale, Postgres with `pg_trgm` for simpler setups
- **Ranking signals**: Always mention multiple ranking factors — text match alone is insufficient
- **Data sync**: If using ES alongside a primary DB, you need **Change Data Capture (CDC)** to keep them in sync
- **Filter sequence**: Apply the most restrictive filter first (usually distance) to shrink the search space
- **Keep it simple**: At Yelp's scale (10M businesses, ~10GB), even Postgres can work. Don't over-engineer!

### What This Toy Does NOT Do

- **No relevance evaluation.** We never measure whether the multi-signal ranking is actually
  *better* — that needs labelled judgements or click data and an offline metric like NDCG.
  Every "this ranking is better" claim in this notebook is an eyeball judgement on 500 rows of
  synthetic data with near-identical descriptions.
- **No personalisation, no query understanding.** Real ranking uses the searcher's history,
  category intent ("pizza" → Restaurants), opening hours, and a learned model on top of BM25.
- **No index/query analyzer tuning.** We use the `standard` analyzer everywhere; production
  autocomplete uses `search_as_you_type` or the completion suggester, as noted above.
- **No incremental sync.** Our reconcile is a full audit-and-repair. Real CDC streams individual
  row changes and only falls back to a full pass as a safety net.

### Architecture Summary

```
User Search Request
        │
        ▼
   ┌─────────┐     cache hit     ┌────────┐
   │  Redis   │◄─────────────────│ API GW │
   │  Cache   │                  └────┬───┘
   └─────────┘                        │ cache miss
                                      ▼
                              ┌───────────────┐
                              │ Elasticsearch  │  ← geo_distance + text match + function_score
                              └───────┬───────┘
                                      │ CDC sync
                                      ▼
                              ┌───────────────┐
                              │  PostgreSQL    │  ← source of truth for businesses & reviews
                              └───────────────┘
```